In [12]:
db = "urcaciv.db"
table = "SRD1CIV" 

#Bibliotecas de Sistema
from datetime import datetime
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing
from sympy import false
from pydoc import text
from sympy import true

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama


def verifica_tipos_de_pedidos(pedido, lista_de_pedidos):
    print("========== Verificando se é um caso de uso conhecido... ==========")

    with sqlite3.connect("movimentos.db") as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT id, resumo FROM pedidos")
        tipos_pedidos = cursor.fetchall()
        tipos_pedidos = "\n".join([f"ID: {id}, Resumo: {resumo}" for id, resumo in tipos_pedidos])        

    pergunta_gemma = "Considere a seguinte lista de pedidos:" \
    f"{lista_de_pedidos}" \
    f"É possível dizer que o pedido '{pedido}' pode ser adequadamente descrito por um item dessa lista?." \
    "Se sim, retorne APENAS o texto EXATO do item correspondente na lista. Se não, não retorne nada."

    resumo = ollama.chat(
        model="cnmoro/gemma3-gaia-ptbr-4b:q8_0",
        messages=[{'role': 'user', 'content': f'{pergunta_gemma}'}],    
    )

    return(resumo['message']['content'])


In [9]:
# Mostra os tipos de pedidos conhecidos'
with sqlite3.connect("movimentos.db") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT id, resumo FROM pedidos")
    tipos_pedidos = cursor.fetchall()
    tipos_pedidos = "\n".join([f"ID: {id}, Resumo: {resumo}" for id, resumo in tipos_pedidos])
    print(tipos_pedidos)

ID: 1, Resumo: Pedido de desistência do processo
ID: 2, Resumo: Informação de que a parte está ciente.
ID: 3, Resumo: Solicita a inclusão da parte executada em cadastros de inadimplentes para fins de cobrança de dívida tributária.
ID: 4, Resumo: Pedido de desconsideração do valor irrisório bloqueado e regular prosseguimento do processo.
ID: 5, Resumo: O município solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução fiscal de baixo valor.
ID: 6, Resumo: Solicita-se a desconsideração do valor irrisório bloqueado e regular prosseguimento do processo.
ID: 7, Resumo: Solicita-se a penhora online via Sisbajud, modalidade reiterada, para identificar e bloquear valores em contas da executada.


In [ ]:
# Apaga por id

idt = input("Digite o ID do registro que deseja remover: ")
# Remover o registro da tabela "perfil" onde o id for igual ao fornecido

# Remover os campos "pet" e "resumo" da tabela "perfil" onde o id for 100

conn = sqlite3.connect(f'{db}')
cursor = conn.cursor()

cursor.execute(f"""
    UPDATE {table}
    SET pet = NULL, resumo = NULL
    WHERE id = {idt}
""")

conn.commit()
conn.close()

In [14]:
# Apaga "pet" se o valor anterior não-nulo for idêntico

conn = sqlite3.connect(f'{db}')
cursor = conn.cursor()

# Busca todos os registros ordenados por id
cursor.execute(f"SELECT id, pet FROM {table} ORDER BY id")
rows = cursor.fetchall()

prev_pet = None
for row in rows:
    current_id, current_pet = row
    if current_pet is not None:
        if prev_pet == current_pet:
            cursor.execute(f"UPDATE {table} SET pet = NULL WHERE id = ?", (current_id,))
        prev_pet = current_pet

conn.commit()
conn.close()

In [ ]:
# Gera um arquivo Excel com os resumos dos pedidos


conn = sqlite3.connect(f'{db}')
cursor = conn.cursor()

cursor.execute(f"SELECT num_processo, tipo, dias, resumo, pet FROM {table} WHERE resumo IS NOT NULL")
result = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]
df_resumo = pd.DataFrame(result, columns=columns)
df_resumo.to_excel(f"{table}.xlsx", index=False)

conn.close()


In [ ]:
# Atualiza o campo "lote" para cada registro com resumo, usando verifica_tipos_de_pedidos

cursor.execute(f"SELECT id, resumo FROM {table} WHERE resumo IS NOT NULL")
rows = cursor.fetchall()

for row in rows:
    registro_id, resumo = row
    print(f"Processando registro ID {registro_id})
    print(f"Resumo: {resumo}")    
    resultado = verifica_tipos_de_pedidos(resumo, tipos_pedidos)
    print(f"Resultado da verificação: {resultado}")
    #if resultado and len(resultado.strip()) > 2:
        #cursor.execute(f"UPDATE {table} SET lote = ? WHERE id = ?", (resultado.strip(), registro_id))

conn.commit()

Processando registro ID: 2208 com resumo: O Município de Sarandi requer que a Vara Judicial da Comarca de Sarandi realize o bloqueio de valores em nome do Executado no Sistema SISBAJUD, com determinação de restrição de transferência, caso a busca inicial seja infrutífera.
========== Verificando se é um caso de uso conhecido... ==========
